In [ ]:
#Підключення бібліотек
from google.colab import files
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
from scipy.stats import pearsonr, spearmanr
from scipy.stats import kruskal
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# Підключення до BigQuery
auth.authenticate_user()

client = bigquery.Client(project="data-analytics-mate")

# SQL-запит
query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `DA.ab_test` ab
    LEFT JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON sp.ga_session_id = ab.ga_session_id
  ),
  sessoins_with_orders AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `DA.order` o
    JOIN session_info si
      ON o.ga_session_id = si.ga_session_id
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group
  ),
  events AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      ep.event_name,
      COUNT(distinct ep.ga_session_id) AS event_cnt
    FROM `DA.event_params` ep
    JOIN session_info si
      ON ep.ga_session_id = si.ga_session_id
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      ep.event_name
  ),
  session AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      COUNT(DISTINCT si.ga_session_id) AS session_cnt
    FROM session_info si
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group
  ),
  account AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `DA.account_session` acs
    JOIN session_info si
      ON acs.ga_session_id = si.ga_session_id
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group
  )
SELECT
  sessoins_with_orders.date,
  sessoins_with_orders.country,
  sessoins_with_orders.device,
  sessoins_with_orders.continent,
  sessoins_with_orders.channel,
  sessoins_with_orders.test,
  sessoins_with_orders.test_group,
  'session with orders' AS event_name,
  sessoins_with_orders.session_with_orders AS value
FROM sessoins_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  new_account_cnt AS value
FROM account
"""

query_job = client.query(query)
results = query_job.result()

In [ ]:
# Створення DataFrame
df = results.to_dataframe()

In [ ]:
#Огляд DataFrame
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,(not set),desktop,Americas,Direct,2,1,new account,1
1,2020-11-01,(not set),desktop,Asia,Direct,2,1,new account,1
2,2020-11-02,(not set),mobile,Asia,Direct,2,2,new account,1
3,2020-11-02,(not set),desktop,Asia,Organic Search,2,1,new account,1
4,2020-11-02,(not set),mobile,Europe,Organic Search,2,2,new account,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


In [ ]:
# Загальна інформація
print('Розмір датасету:', df.shape)

# Кількість унікальних значень
print('\nКількість тестів:', df['test'].nunique())
print('Тести:', sorted(df['test'].unique()))

print('\nГрупи:', sorted(df['test_group'].unique()))
print('Пристрої:', df['device'].nunique())
print('Країни:', df['country'].nunique())
print('Канали:', df['channel'].nunique())

# Кількість кожної події
print('\nТипи подій:')
print(df['event_name'].value_counts())

# Період даних
print('\nПеріод даних:')
print('Від:', df['date'].min())
print('До:', df['date'].max())

# Перевірка дублікатів
print('\nКількість дублікатів:', df.duplicated().sum())

Розмір датасету: (800996, 9)

Кількість тестів: 4
Тести: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Групи: [np.int64(1), np.int64(2)]
Пристрої: 3
Країни: 108
Канали: 5

Типи подій:
event_name
session                107210
session_start          106242
page_view              101907
user_engagement         94520
first_visit             81621
scroll                  73643
view_promotion          61695
view_item               44869
session with orders     25892
new account             22389
view_search_results     14310
add_shipping_info       11961
begin_checkout          11959
select_item             11816
add_to_cart             11762
select_promotion         9044
add_payment_info         8048
click                    1997
view_item_list            111
Name: count, dtype: int64

Період даних:
Від: 2020-11-01
До: 2021-01-27

Кількість дублікатів: 0


##Розрахунок статистичної значущості для чотирьох метрик:

In [ ]:
# Створюємо словник метрик
metrics = {
    'add_payment_info / session': 'add_payment_info',
    'add_shipping_info / session': 'add_shipping_info',
    'begin_checkout / session': 'begin_checkout',
    'new_accounts / session': 'new account'
}

###Розраховуємо статистичну значущість для всіх тестів

In [ ]:
# Список усіх тестів
tests = sorted(df['test'].unique())

# Список для збереження результатів
results = []

# Цикл по тестах
for test_id in tests:

    # Фільтруємо поточний тест
    test_df = df[df['test'] == test_id]

    # Кількість сесій у кожній групі
    sessions = (
        test_df[test_df['event_name'] == 'session']
        .groupby('test_group')['value']
        .sum()
    )

    # Цикл по метриках
    for metric, event in metrics.items():

        # Кількість конверсій у кожній групі
        conversions = (
            test_df[test_df['event_name'] == event]
            .groupby('test_group')['value']
            .sum()
        )

        # Кількість конверсій: Control та Test
        count = [
            conversions.get(1, 0),
            conversions.get(2, 0)
        ]

        # Кількість сесій: Control та Test
        nobs = [
            sessions.get(1, 0),
            sessions.get(2, 0)
        ]

        # Z-test
        z_stat, p_value = proportions_ztest(count, nobs)

        # Конверсії
        control_cr = count[0] / nobs[0]
        test_cr = count[1] / nobs[1]

        # Зберігаємо результати
        results.append({
            'test': test_id,
            'metric': metric,
            'control_count': count[0],
            'test_count': count[1],
            'control_sessions': nobs[0],
            'test_sessions': nobs[1],
            'control_cr': control_cr,
            'test_cr': test_cr,
            'z_stat': z_stat,
            'p_value': p_value,
            'is_significant': p_value < 0.05
        })

# Створюємо фінальну таблицю
results_df = pd.DataFrame(results)

results_df.round(4)

,test,metric,control_count,test_count,control_sessions,test_sessions,control_cr,test_cr,z_stat,p_value,is_significant
0,1,add_payment_info / session,964,983,45362,45193,0.0213,0.0218,-0.5186,0.6041,False
1,1,add_shipping_info / session,1860,1992,45362,45193,0.0410,0.0441,-2.2919,0.0219,True
2,1,begin_checkout / session,1862,1992,45362,45193,0.0410,0.0441,-2.2585,0.0239,True
3,1,new_accounts / session,3823,3681,45362,45193,0.0843,0.0815,1.5429,0.1229,False
4,2,add_payment_info / session,1096,1125,50637,50244,0.0216,0.0224,-0.8079,0.4192,False
5,2,add_shipping_info / session,2194,2136,50637,50244,0.0433,0.0425,0.6389,0.5229,False
6,2,begin_checkout / session,2195,2137,50637,50244,0.0433,0.0425,0.6387,0.5230,False
7,2,new_accounts / session,4165,4184,50637,50244,0.0823,0.0833,-0.5888,0.5560,False
8,3,add_payment_info / session,1771,1788,70047,70439,0.0253,0.0254,-0.1200,0.9045,False
9,3,add_shipping_info / session,2915,2889,70047,70439,0.0416,0.0410,0.5657,0.5716,False


##Розрахунку в розрізі пристроїв (device).

In [ ]:
# Список пристроїв
devices = sorted(df['device'].unique())

print(devices)

['desktop', 'mobile', 'tablet']


###Розраховуємо статистичну значущість

In [ ]:
# Список для збереження результатів
device_results = []

# Цикл по тестах
for test_id in tests:

    # Фільтруємо поточний тест
    test_df = df[df['test'] == test_id]

    # Цикл по пристроях
    for device in devices:

        # Фільтруємо поточний пристрій
        device_df = test_df[test_df['device'] == device]

        # Кількість сесій у кожній групі
        sessions = (
            device_df[device_df['event_name'] == 'session']
            .groupby('test_group')['value']
            .sum()
        )

        # Цикл по метриках
        for metric, event in metrics.items():

            # Кількість конверсій у кожній групі
            conversions = (
                device_df[device_df['event_name'] == event]
                .groupby('test_group')['value']
                .sum()
            )

            # Конверсії та сесії для Control і Test
            count = [
                conversions.get(1, 0),
                conversions.get(2, 0)
            ]

            nobs = [
                sessions.get(1, 0),
                sessions.get(2, 0)
            ]

            # Перевіряємо наявність сесій
            if nobs[0] == 0 or nobs[1] == 0:
                continue

            # Z-test
            z_stat, p_value = proportions_ztest(count, nobs)

            # Розрахунок конверсій
            control_cr = count[0] / nobs[0]
            test_cr = count[1] / nobs[1]

            # Зберігаємо результат
            device_results.append({
                'test': test_id,
                'device': device,
                'metric': metric,
                'control_count': count[0],
                'test_count': count[1],
                'control_sessions': nobs[0],
                'test_sessions': nobs[1],
                'control_cr': control_cr,
                'test_cr': test_cr,
                'z_stat': z_stat,
                'p_value': p_value,
                'is_significant': p_value < 0.05
            })

# Створюємо DataFrame
device_results_df = pd.DataFrame(device_results)

device_results_df.round(4)

,test,device,metric,control_count,test_count,control_sessions,test_sessions,control_cr,test_cr,z_stat,p_value,is_significant
0,1,desktop,add_payment_info / session,544,562,26467,26417,0.0206,0.0213,-0.5788,0.5627,False
1,1,desktop,add_shipping_info / session,1049,1170,26467,26417,0.0396,0.0443,-2.6698,0.0076,True
2,1,desktop,begin_checkout / session,1051,1170,26467,26417,0.0397,0.0443,-2.6253,0.0087,True
3,1,desktop,new_accounts / session,2203,2147,26467,26417,0.0832,0.0813,0.8212,0.4115,False
4,1,mobile,add_payment_info / session,397,403,17896,17767,0.0222,0.0227,-0.3180,0.7505,False
5,1,mobile,add_shipping_info / session,768,785,17896,17767,0.0429,0.0442,-0.5869,0.5573,False
6,1,mobile,begin_checkout / session,768,785,17896,17767,0.0429,0.0442,-0.5869,0.5573,False
7,1,mobile,new_accounts / session,1530,1441,17896,17767,0.0855,0.0811,1.4995,0.1337,False
8,1,tablet,add_payment_info / session,23,18,999,1009,0.0230,0.0178,0.8212,0.4115,False
9,1,tablet,add_shipping_info / session,43,37,999,1009,0.0430,0.0367,0.7301,0.4654,False


In [ ]:
print('Кількість рядків:', len(device_results_df))
print('Кількість тестів:', device_results_df['test'].nunique())
print('Кількість пристроїв:', device_results_df['device'].nunique())
print('Кількість метрик:', device_results_df['metric'].nunique())

Кількість рядків: 48
Кількість тестів: 4
Кількість пристроїв: 3
Кількість метрик: 4


##Розрахунку в розрізі континентів.

In [ ]:
# Створюємо список континентів
continents = df['continent'].unique()

# Створюємо список для результатів
continent_results = []

# Цикл за континентами
for continent in continents:

    # Виключаємо невідомий континент
    if continent == '(not set)':
        continue

    # Фільтруємо дані за континентом
    continent_df = df[df['continent'] == continent]

    # Розраховуємо метрики для кожного тесту
    for test_id in tests:

        test_df = continent_df[continent_df['test'] == test_id]

        # Створюємо список континентів без невідомих значень
continents = df.loc[
    df['continent'] != '(not set)', 'continent'
].dropna().unique()

# Створюємо список для результатів
continent_results = []

# Цикл за континентами
for continent in continents:

    # Фільтруємо дані за континентом
    continent_df = df[df['continent'] == continent]

    # Цикл за тестами
    for test_id in tests:

        test_df = continent_df[continent_df['test'] == test_id]

        # Кількість сесій у кожній групі
        sessions = (
            test_df[test_df['event_name'] == 'session']
            .groupby('test_group')['value']
            .sum()
        )

        # Цикл за метриками
        for metric, event in metrics.items():

            # Кількість конверсій у кожній групі
            conversions = (
                test_df[test_df['event_name'] == event]
                .groupby('test_group')['value']
                .sum()
            )

            # Контрольна та тестова групи
            count = [
                conversions.get(1, 0),
                conversions.get(2, 0)
            ]

            nobs = [
                sessions.get(1, 0),
                sessions.get(2, 0)
            ]

            # Перевіряємо, чи є сесії в обох групах
            if nobs[0] == 0 or nobs[1] == 0:
                continue

            # Розраховуємо конверсії
            control_cr = count[0] / nobs[0]
            test_cr = count[1] / nobs[1]

            # Z-test
            z_stat, p_value = proportions_ztest(count, nobs)

            # Зберігаємо результати
            continent_results.append({
                'continent': continent,
                'test': test_id,
                'metric': metric,
                'control_count': count[0],
                'test_count': count[1],
                'control_sessions': nobs[0],
                'test_sessions': nobs[1],
                'control_cr': control_cr,
                'test_cr': test_cr,
                'z_stat': z_stat,
                'p_value': p_value,
                'is_significant': p_value < 0.05
            })

# Створюємо фінальну таблицю
continent_results_df = pd.DataFrame(continent_results)

continent_results_df.round(4)

,continent,test,metric,control_count,test_count,control_sessions,test_sessions,control_cr,test_cr,z_stat,p_value,is_significant
0,Americas,1,add_payment_info / session,543,533,25164,24769,0.0216,0.0215,0.0459,0.9634,False
1,Americas,1,add_shipping_info / session,1038,1106,25164,24769,0.0412,0.0447,-1.8756,0.0607,False
2,Americas,1,begin_checkout / session,1040,1106,25164,24769,0.0413,0.0447,-1.8310,0.0671,False
3,Americas,1,new_accounts / session,2165,1971,25164,24769,0.0860,0.0796,2.6187,0.0088,True
4,Americas,2,add_payment_info / session,594,642,27902,27772,0.0213,0.0231,-1.4637,0.1433,False
...,...,...,...,...,...,...,...,...,...,...,...,...
75,Oceania,3,new_accounts / session,47,51,684,763,0.0687,0.0668,0.1415,0.8875,False
76,Oceania,4,add_payment_info / session,25,22,1095,1069,0.0228,0.0206,0.3592,0.7195,False
77,Oceania,4,add_shipping_info / session,32,27,1095,1069,0.0292,0.0253,0.5665,0.5711,False
78,Oceania,4,begin_checkout / session,32,27,1095,1069,0.0292,0.0253,0.5665,0.5711,False


###Розрахунок за каналами (Channel)

In [ ]:
# Створюємо список каналів
channels = df['channel'].dropna().unique()

# Створюємо список для результатів
channel_results = []

# Цикл за каналами
for channel in channels:

    # Фільтруємо дані за каналом
    channel_df = df[df['channel'] == channel]

    # Цикл за тестами
    for test_id in tests:

        test_df = channel_df[channel_df['test'] == test_id]

        # Кількість сесій у кожній групі
        sessions = (
            test_df[test_df['event_name'] == 'session']
            .groupby('test_group')['value']
            .sum()
        )

        # Цикл за метриками
        for metric, event in metrics.items():

            # Кількість конверсій у кожній групі
            conversions = (
                test_df[test_df['event_name'] == event]
                .groupby('test_group')['value']
                .sum()
            )

            # Кількість конверсій Control і Test
            count = [
                conversions.get(1, 0),
                conversions.get(2, 0)
            ]

            # Кількість сесій Control і Test
            nobs = [
                sessions.get(1, 0),
                sessions.get(2, 0)
            ]

            # Перевіряємо наявність сесій
            if nobs[0] == 0 or nobs[1] == 0:
                continue

            # Розраховуємо конверсії
            control_cr = count[0] / nobs[0]
            test_cr = count[1] / nobs[1]

            # Розраховуємо статистичну значущість
            z_stat, p_value = proportions_ztest(count, nobs)

            # Зберігаємо результати
            channel_results.append({
                'channel': channel,
                'test': test_id,
                'metric': metric,
                'control_count': count[0],
                'test_count': count[1],
                'control_sessions': nobs[0],
                'test_sessions': nobs[1],
                'control_cr': control_cr,
                'test_cr': test_cr,
                'z_stat': z_stat,
                'p_value': p_value,
                'is_significant': p_value < 0.05
            })

# Створюємо DataFrame
channel_results_df = pd.DataFrame(channel_results)

channel_results_df.round(4)

,channel,test,metric,control_count,test_count,control_sessions,test_sessions,control_cr,test_cr,z_stat,p_value,is_significant
0,Direct,1,add_payment_info / session,205,228,10691,10361,0.0192,0.0220,-1.4466,0.1480,False
1,Direct,1,add_shipping_info / session,419,438,10691,10361,0.0392,0.0423,-1.1313,0.2579,False
2,Direct,1,begin_checkout / session,420,438,10691,10361,0.0393,0.0423,-1.0964,0.2729,False
3,Direct,1,new_accounts / session,913,850,10691,10361,0.0854,0.0820,0.8800,0.3789,False
4,Direct,2,add_payment_info / session,247,242,11820,11670,0.0209,0.0207,0.0858,0.9316,False
...,...,...,...,...,...,...,...,...,...,...,...,...
75,Paid Search,3,new_accounts / session,1576,1553,18763,18744,0.0840,0.0829,0.3999,0.6892,False
76,Paid Search,4,add_payment_info / session,406,404,28974,28739,0.0140,0.0141,-0.0459,0.9634,False
77,Paid Search,4,add_shipping_info / session,605,650,28974,28739,0.0209,0.0226,-1.4302,0.1527,False
78,Paid Search,4,begin_checkout / session,605,650,28974,28739,0.0209,0.0226,-1.4302,0.1527,False


##Обєднуємо нашу таблицю з чотирма метриками та трьома розрізами

In [ ]:
# Загальні результати
total_df = results_df.copy()
total_df['dimension'] = 'total'
total_df['dimension_value'] = 'All'

# Результати за пристроями
device_df = device_results_df.copy()
device_df['dimension'] = 'device'
device_df = device_df.rename(columns={'device': 'dimension_value'})

# Результати за континентами
continent_df = continent_results_df.copy()
continent_df['dimension'] = 'continent'
continent_df = continent_df.rename(columns={'continent': 'dimension_value'})

# Результати за каналами
channel_df = channel_results_df.copy()
channel_df['dimension'] = 'channel'
channel_df = channel_df.rename(columns={'channel': 'dimension_value'})

In [ ]:
# Об'єднуємо всі результати
final_df = pd.concat(
    [total_df, device_df, continent_df, channel_df],
    ignore_index=True
)

# Переглядаємо результат
final_df.head(20)

,test,metric,control_count,test_count,control_sessions,test_sessions,control_cr,test_cr,z_stat,p_value,is_significant,dimension,dimension_value
0,1,add_payment_info / session,964,983,45362,45193,0.021251,0.021751,-0.518551,0.604074,False,total,All
1,1,add_shipping_info / session,1860,1992,45362,45193,0.041003,0.044078,-2.291930,0.021910,True,total,All
2,1,begin_checkout / session,1862,1992,45362,45193,0.041048,0.044078,-2.258498,0.023915,True,total,All
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451,1.542883,0.122859,False,total,All
4,2,add_payment_info / session,1096,1125,50637,50244,0.021644,0.022391,-0.807895,0.419151,False,total,All
5,2,add_shipping_info / session,2194,2136,50637,50244,0.043328,0.042513,0.638943,0.522860,False,total,All
6,2,begin_checkout / session,2195,2137,50637,50244,0.043348,0.042532,0.638682,0.523030,False,total,All
7,2,new_accounts / session,4165,4184,50637,50244,0.082252,0.083274,-0.588793,0.556000,False,total,All
8,3,add_payment_info / session,1771,1788,70047,70439,0.025283,0.025384,-0.120029,0.904461,False,total,All
9,3,add_shipping_info / session,2915,2889,70047,70439,0.041615,0.041014,0.565667,0.571620,False,total,All


In [ ]:
print('Кількість рядків:', len(final_df))
print('Кількість тестів:', final_df['test'].nunique())
print('Кількість метрик:', final_df['metric'].nunique())

print('\nРозподіл за розрізами:')
print(final_df['dimension'].value_counts())

print('\nКількість колонок:', len(final_df.columns))

Кількість рядків: 224
Кількість тестів: 4
Кількість метрик: 4

Розподіл за розрізами:
dimension
channel      80
continent    80
device       48
total        16
Name: count, dtype: int64

Кількість колонок: 13


In [ ]:
# Перевіряємо дублікати
duplicates = final_df.duplicated(
    subset=['test', 'metric', 'dimension', 'dimension_value']
).sum()

print('Кількість дублікатів:', duplicates)

Кількість дублікатів: 0


In [ ]:
# Перевіряємо пропущені значення
final_df.isnull().sum()

,0
test,0
metric,0
control_count,0
test_count,0
control_sessions,0
test_sessions,0
control_cr,0
test_cr,0
z_stat,0
p_value,0


In [ ]:
# Перевіряємо кількість метрик у кожному розрізі
final_df.groupby('dimension')['metric'].nunique()

,metric
dimension,
channel,4
continent,4
device,4
total,4


##Дадаємо ще одну колонку до таблиці - відносна зміна конверсії тестової групи порівняно з контрольною, у відсотках.

In [ ]:
# Розраховуємо зміну конверсії у відсотках
final_df.insert(
    8,
    'metric_change',
    ((final_df['test_cr'] - final_df['control_cr']) / final_df['control_cr']) * 100
)

# Перевіряємо результат
final_df.head()

,test,metric,control_count,test_count,control_sessions,test_sessions,control_cr,test_cr,metric_change,z_stat,p_value,is_significant,dimension,dimension_value
0,1,add_payment_info / session,964,983,45362,45193,0.021251,0.021751,2.352276,-0.518551,0.604074,False,total,All
1,1,add_shipping_info / session,1860,1992,45362,45193,0.041003,0.044078,7.497264,-2.291930,0.021910,True,total,All
2,1,begin_checkout / session,1862,1992,45362,45193,0.041048,0.044078,7.381800,-2.258498,0.023915,True,total,All
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451,-3.354299,1.542883,0.122859,False,total,All
4,2,add_payment_info / session,1096,1125,50637,50244,0.021644,0.022391,3.448865,-0.807895,0.419151,False,total,All


In [ ]:
#Завантажуємо файл для подальшої роботи з ним в Tableau
final_df.to_csv('ab_test_results.csv', index=False, encoding='utf-8-sig')

final_df.to_excel('ab_test_results.xlsx', index=False)

# Загальний висновок

У межах проєкту було проведено аналіз результатів чотирьох A/B-тестів з метою оцінки впливу тестових змін на поведінку користувачів та конверсійні показники.

Для аналізу було отримано дані з BigQuery, проведено їх дослідження та підготовку за допомогою Python і Pandas. Було розраховано чотири основні конверсійні метрики та перевірено статистичну значущість відмінностей між контрольною і тестовою групами за допомогою Z-тесту.

Додатково результати було досліджено в розрізі пристроїв, континентів і каналів залучення користувачів, щоб визначити можливі відмінності в поведінці окремих сегментів.

За результатами аналізу виявлено статистично значуще покращення двох метрик у тесті №1 — add_shipping_info та begin_checkout. Водночас у тесті №4 зафіксовано статистично значуще зниження конверсії new_accounts. Для інших метрик на загальному рівні статистично значущих змін не виявлено.

Підготовлені результати було об'єднано в єдину таблицю та експортовано для подальшої візуалізації в Tableau. Це дозволило створити інтерактивний дашборд для зручного порівняння результатів тестів і дослідження окремих сегментів користувачів.

Таким чином, аналіз допоміг визначити, які зміни показали позитивний або негативний результат, а також продемонстрував важливість перевірки статистичної значущості перед ухваленням продуктових рішень.

## General Conclusion

In this project, we analyzed the results of four A/B tests. The main goal was to understand how the changes affected user behavior and conversion rates.

First, we collected data from BigQuery and prepared it for analysis using Python and Pandas. We calculated four main conversion metrics and checked the differences between the Control and Test groups using the Z-test.

We also analyzed the results by different user segments, including devices, continents, and traffic channels. This helped us understand how different groups of users behaved.

The analysis showed a statistically significant improvement in two metrics in Test 1: add_shipping_info and begin_checkout. However, Test 4 showed a statistically significant decrease in the new_accounts conversion rate. No statistically significant changes were found for the other metrics at the overall level.

After the analysis, we combined the results into one table and exported the data for further visualization in Tableau. This allowed us to create an interactive dashboard to compare test results and explore different user segments.

As a result, the analysis helped us identify which changes had positive or negative results. It also showed why statistical significance is important when making product decisions.

#Посилання на інтерактивний дашборд
https://public.tableau.com/app/profile/.32455230/viz/PortfolioProject2_17940407236070/ABtest